# web scraping
- issue: lifting cast is a dynamically rendered website with data constantly being updated with new competition data

- decision: using a webscraping library which allows us to scrape dynamically rendered like playwright to search for the competition data which can be downloaded from a button

- group: Data extraction

- assumptions: lifting cast should should allow us to download the data from the button

- constraints: buttons are not clearly labeled, button could fail to download data, could get blocked or banned for web scraping

- position: we could also use other libraries like selenium, crawl4ai, scrapy to scrape dynamic websites. the data can also be scraped from the tables which are not clearly labeled but can be parsed.

- implications: a playwright will need to use a machine with python 3.7+ windows 11+ , chromium, 

---


In [1]:
# libraries

# web scraping + selenium specific libraires
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# standard libraries
from datetime import datetime, timedelta
import os, random, re
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

# database module
from db_init import init_db

# scraping lifting cast for competitions
---

In [2]:
USER_AGENTS = [
    # Chrome 120+ Windows
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
]

def get_random_user():
    return random.choice(USER_AGENTS)

In [3]:
# global variables
url = "https://liftingcast.com"
COMPETITION_ROW = "div.meets-table-table-wrapper table.table tbody tr"
WAIT_TIME = 60
USER_AGENT = get_random_user()
COMPETITION_LINKS = "div.meets-table-table-wrapper table.table a[href]"

# create folder to store competition data
DOWNLOAD_DIR = os.path.abspath(os.path.join(os.getcwd(), "data"))
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print("Current working directory:", os.getcwd())
print("Download directory:", DOWNLOAD_DIR)
print("Directory exists:", os.path.exists(DOWNLOAD_DIR))

# pulls driver to avoid downloading manualy
options = Options()

# headless driver + reliability
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument(f"--user-agent={USER_AGENT}")

# allowing for downloads early
options.add_experimental_option("prefs", {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
})

# creating chrome driver
driver = webdriver.Chrome(options=options)

# remove webdriver flag
driver.execute_script(
    "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
)

wait = WebDriverWait(driver,WAIT_TIME)

# allow headless downloads
driver.execute_cdp_cmd("Page.setDownloadBehavior", {
    "behavior": "allow",
    "downloadPath": DOWNLOAD_DIR
})

Current working directory: c:\Users\josed\projects\webscraping_liftingcast\notebooks
Download directory: c:\Users\josed\projects\webscraping_liftingcast\notebooks\data
Directory exists: True


{}

In [4]:
# using the driver we automate and scrape the lifting cast competition page
driver.get(url=url)

# wait for the table to exists with rows
wait.until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR,COMPETITION_ROW)))

# wait for the table to be populated
wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, COMPETITION_LINKS)) >= 5)

# store out html, parse, and store as rows of competitions
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
rows = soup.find_all('tr')

# quit driver to save resources
driver.quit()

### Storing raw data
---

In [5]:
# display raw data
rows

[<tr><td colspan="3"><div class="table-title"><a href="#upcoming-meets" id="upcoming-meets">Upcoming Meets</a><input placeholder="Search..." type="text" value=""/></div></td></tr>,
 <tr><th style="width: 70%;">Name</th><th style="width: 30%;">Date</th></tr>,
 <tr><td><a href="/meets/m99nrpg1qa1o/results">Mega Nacional DIA 6</a></td><td>03/18/2026</td></tr>,
 <tr><td><a href="/meets/m0c7z88bas1x/results">2026 Powerlifting America MPG Madness</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/m56t1rl0zpyd/results">2026 Special Olympics Illinois Region D Powerlifting Qualifier</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/ms6o5pezlyu1/results">2026 USAPL CT State Championships (CT-2026-04)</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/m2ctjv3jdstt/results">2026 USAPL Hammer Forged Classic</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/mjk1hufxj96b/results">2026 USAPL MD State Championship (MD-2026-01)</a></td><td>03/21/2026</td></tr>,
 <tr><td

In [6]:
# remove the first 2 rows as they arent entries
rows = rows[2:]
# display the first 5 rows to check
rows[:5]

[<tr><td><a href="/meets/m99nrpg1qa1o/results">Mega Nacional DIA 6</a></td><td>03/18/2026</td></tr>,
 <tr><td><a href="/meets/m0c7z88bas1x/results">2026 Powerlifting America MPG Madness</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/m56t1rl0zpyd/results">2026 Special Olympics Illinois Region D Powerlifting Qualifier</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/ms6o5pezlyu1/results">2026 USAPL CT State Championships (CT-2026-04)</a></td><td>03/21/2026</td></tr>,
 <tr><td><a href="/meets/m2ctjv3jdstt/results">2026 USAPL Hammer Forged Classic</a></td><td>03/21/2026</td></tr>]

In [7]:
# we want to store the data in a database so we will format the date early as yyyy-mm-dd
def parse_mmddyyyy(value):
    """
    Accepts: '02/26/2026', '2/6/2026', or None/''.
    Returns: datetime.date or None
    """
    if value is None:
        return None

    s = str(value).strip()
    if not s:
        return None

    # Normalize any separators like 02-26-2026 -> 02/26/2026
    s = re.sub(r"[-.]", "/", s)

    # Strict parse (month/day/year)
    try:
        return datetime.strptime(s, "%m/%d/%Y").date()
    except ValueError:
        return None  # or raise, depending on how strict you want it


def to_iso_date(value):
    """Returns 'YYYY-MM-DD' or None"""
    d = parse_mmddyyyy(value)
    return d.isoformat() if d else None

In [8]:
# accessing the link, name, and data elements
row = rows[0]
 # seporating row into its table data components
tds = row.find_all('td')
        
# storing link thorugh the href
link = tds[0].a.get('href')
# storing contents from a tag
name = tds[0].a.contents[0]
# storing date from the second item
raw_date = tds[1].contents[0]
date = to_iso_date(raw_date)

print(f"url:  {link}\nname: {name}\ndate: {date}")

url:  /meets/m99nrpg1qa1o/results
name: Mega Nacional DIA 6
date: 2026-03-18


In [9]:
def date_in_days(days:int):
    '''
    using datetime and delta we calculate the future date from the number of days were looking for
    '''
    return (datetime.today() + timedelta(days=days)).date()

In [10]:
# each row contains 2 table data components

# td 1 contains a link to the competition in the a tag and the competition name
## href contains link
## contents contain competition name

# td 2 contains the date of the competition in the format mm,dd,yyyy
## stored in contents
# getting the date in a week to use to remove 

DATE_RANGE = 7
lastDate = date_in_days(DATE_RANGE)

compList = []

for competition in rows:
    try:
        # seporating row into its table data components
        tds = competition.find_all('td')
        
        # extract data from each row
        link = tds[0].a.get('href')
        name = tds[0].a.contents[0]
        raw_date = tds[1].contents[0]
        date = datetime.strptime(raw_date, "%m/%d/%Y").date()

        # converting date to sqlite compatible version
        formated_date = to_iso_date(raw_date)

        # check if data is within the 7 days
        if  date <= lastDate:
            comp = {'name':name,
                    'link':url + link,
                    'date':formated_date}
            compList.append(comp)
        else:
            break
    except :
        print("ERROR row data:", competition)
        break

In [11]:
compList

[{'name': 'Mega Nacional DIA 6',
  'link': 'https://liftingcast.com/meets/m99nrpg1qa1o/results',
  'date': '2026-03-18'},
 {'name': '2026 Powerlifting America MPG Madness',
  'link': 'https://liftingcast.com/meets/m0c7z88bas1x/results',
  'date': '2026-03-21'},
 {'name': '2026 Special Olympics Illinois Region D Powerlifting Qualifier',
  'link': 'https://liftingcast.com/meets/m56t1rl0zpyd/results',
  'date': '2026-03-21'},
 {'name': '2026 USAPL CT State Championships (CT-2026-04)',
  'link': 'https://liftingcast.com/meets/ms6o5pezlyu1/results',
  'date': '2026-03-21'},
 {'name': '2026 USAPL Hammer Forged Classic',
  'link': 'https://liftingcast.com/meets/m2ctjv3jdstt/results',
  'date': '2026-03-21'},
 {'name': '2026 USAPL MD State Championship (MD-2026-01)',
  'link': 'https://liftingcast.com/meets/mjk1hufxj96b/results',
  'date': '2026-03-21'},
 {'name': '👑2026👑 USAPL Utah State Championships (UT-2026-02)',
  'link': 'https://liftingcast.com/meets/mtcogci9dc0h/results',
  'date': '20

In [12]:
competition = compList[0]
print(competition['name'])
print(competition['link'])
print(competition['date'])

Mega Nacional DIA 6
https://liftingcast.com/meets/m99nrpg1qa1o/results
2026-03-18


### connecting to SQLite database 
---

inserting competition list

In [29]:
def insert_competitions(conn, compList):
    sql = """
    INSERT INTO competitions (comp_name, comp_url, comp_date)
    VALUES (?, ?, ?)
    """

    rows = []
    for c in compList:
        rows.append((
            c["name"].strip(),
            c["link"].strip(),
            c["date"].strip()   # datetime.date -> 'YYYY-MM-DD'
        ))

    with conn:
        conn.executemany(sql, rows)

    return len(rows)

In [30]:
conn = init_db()
n = insert_competitions(conn, compList)
print("inserted/updated : ", n)
conn.close()

inserted/updated :  28


In [32]:
def get_all_competitions(conn):
    return conn.execute("""
        SELECT *
        FROM competitions
    """).fetchall()

conn = init_db()
competitions = get_all_competitions(conn)
conn.close()
for comp in competitions:
    print(dict(comp))

{'comp_id': 1, 'comp_name': 'Mega Nacional DIA 6', 'comp_url': 'https://liftingcast.com/meets/m99nrpg1qa1o/results', 'comp_date': '2026-03-18'}
{'comp_id': 2, 'comp_name': '2026 Powerlifting America MPG Madness', 'comp_url': 'https://liftingcast.com/meets/m0c7z88bas1x/results', 'comp_date': '2026-03-21'}
{'comp_id': 3, 'comp_name': '2026 Special Olympics Illinois Region D Powerlifting Qualifier', 'comp_url': 'https://liftingcast.com/meets/m56t1rl0zpyd/results', 'comp_date': '2026-03-21'}
{'comp_id': 4, 'comp_name': '2026 USAPL CT State Championships (CT-2026-04)', 'comp_url': 'https://liftingcast.com/meets/ms6o5pezlyu1/results', 'comp_date': '2026-03-21'}
{'comp_id': 5, 'comp_name': '2026 USAPL Hammer Forged Classic', 'comp_url': 'https://liftingcast.com/meets/m2ctjv3jdstt/results', 'comp_date': '2026-03-21'}
{'comp_id': 6, 'comp_name': '2026 USAPL MD State Championship (MD-2026-01)', 'comp_url': 'https://liftingcast.com/meets/mjk1hufxj96b/results', 'comp_date': '2026-03-21'}
{'comp_id

### TODO
---
- remove competitions and athletes associated with the competition past competition dates
- update csv name whenever it is downloaded from the website